<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_35_django_forms/note_lesson_35_forms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Урок 35 — Django: форми, CRUD і PRG

Після уроку 34 нотатки можна створювати лише в адмінці. Сьогодні проєкт нотаток отримує **форми**: користувач сайту створює, змінює й видаляє нотатки сам. Усе — на справжньому коді уроку: `django_bootstrap_project` (рефакторинг 1) працює прямо в ноутбуці, а `crispy_notes_project` (рефакторинг 2) — в окремому процесі.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкт у ноутбуці, свіжа база `seed()` | 0.1 | |
| 1 | `ModelForm`: поля з моделі, `is_valid()`, `cleaned_data`, `errors`, `Meta.fields`, `save(commit=False)`, власні перевірки `clean_<поле>()` і `clean()`, звичайна `Form` для пошуку | 1.1–1.10 | **Задача 1** |
| 2 | view → шаблон: контекст, ланцюжок шаблонів, `{% extends %}` у мініатюрі, екранування, `reverse`, `404` | 2.1–2.6 | |
| 3 | створення: три гілки `note_create`, PRG, `follow=True`, повідомлення й `MESSAGE_TAGS` | 3.1–3.6 | |
| 4 | редагування й видалення: `instance=`, `changed_data`, підтвердження, `@require_POST` | 4.1–4.5 | **Задача 2** |
| 5 | CSRF: без токена — `403`, з токеном — `302` | 5.1–5.2 | |
| 6 | знайди помилку: форма без PRG | 6.1 | **Задача 3** |
| 7 | рефакторинг 2: `{% crispy form %}`, `@login_required`, фільтр записників, context processor, тихий баг `is_pinned` | 7.1–7.6 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Рядки на кшталт `Not Found: /notes/999/` чи `Forbidden (CSRF cookie not set.)` у виводі — це не `print`, а журнал Django: кожну відповідь 4xx він записує окремо.
- Теорія, діфи рефакторингів і скріншоти — у книзі курсу: [Урок 35](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/); поглиблено — кроки 2 і 4 [Django-книги](https://nikoriakviktor.github.io/notes_chat_app/tutorials/).

---
## 0. Проєкт у ноутбуці

Проєкт — `django_bootstrap_project` з уроку: `NoteForm` (`hello_app/forms.py`), CRUD-views за шаблоном PRG і шаблони на Bootstrap. Це стартовий код, який на сторінці уроку розібрано як рефакторинг проєкту уроку 34. У Colab клітинка нижче завантажить лише папку уроку (обидва проєкти) і встановить залежності, яких бракує.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_35_django_forms"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if os.path.basename(os.getcwd()) == "django_bootstrap_project":   # клітинку запустили вдруге
    os.chdir("..")
if not os.path.isdir("django_bootstrap_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))
LESSON_DIR = os.getcwd()

packages = {"django": "Django>=5.2,<6", "django_bootstrap5": "django-bootstrap5", "unfold": "django-unfold",
            "debug_toolbar": "django-debug-toolbar", "crispy_forms": "django-crispy-forms",
            "crispy_bootstrap5": "crispy-bootstrap5"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

os.chdir("django_bootstrap_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
from collections import Counter

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 34)
django.setup()

from django import forms
from django.contrib.messages import constants as message_levels
from django.contrib.messages.storage.base import Message
from django.http import Http404, HttpResponse, QueryDict
from django.shortcuts import get_object_or_404, redirect, render
from django.template import Context, Engine
from django.test import Client, RequestFactory
from django.test.utils import setup_test_environment
from django.urls import resolve, reverse
from django.views.decorators.http import require_POST

from hello_app.forms import NoteForm
from hello_app.models import Note

setup_test_environment()                               # хост testserver і контекст шаблонів у відповідях
client = Client()                                      # «браузер» без сервера
factory = RequestFactory()                             # окремий запит для прямого виклику view
print("Django", django.get_version(), "| проєкт:", os.path.basename(os.getcwd()))

### Приклад 0.1. Свіжа база одним викликом

Розділи ноутбука змінюють базу: створюють, редагують, видаляють. Щоб кожен розділ починався з відомого стану, опишемо функцію `seed()`: вона видаляє всі нотатки й створює три початкові. Повертає словник «заголовок → нотатка», щоб не шукати їх щоразу.

In [ ]:
START_NOTES = [
    ("Купити молоко", "2 л"),
    ("Вивчити Django Forms", "ModelForm, is_valid, PRG"),
    ("Пароль від Wi-Fi", "coffee2026"),
]


def seed():
    """Свіжий стан бази: лише три початкові нотатки (у порядку створення)."""
    Note.objects.all().delete()
    return {title: Note.objects.create(title=title, content=content) for title, content in START_NOTES}

In [ ]:
notes = seed()
print("нотаток:", Note.objects.count())
print("у порядку моделі:", list(Note.objects.values_list("title", flat=True)))

Порядок — від нової до старої: у моделі `Meta.ordering = ['-created_at']`. «Пароль від Wi-Fi» створено останнім, тож він перший.

---
## 1. `ModelForm`

Форма робить для сторінки три речі: **показує** поля (HTML), **перевіряє** те, що прийшло з браузера, і **перетворює** рядки в значення Python. `ModelForm` бере поля й перевірки з моделі:

```
Note (models.py)                         NoteForm (forms.py)
title   = CharField(max_length=200)  →   title:   обов'язкове, ≤ 200 символів, <input maxlength="200">
content = TextField(blank=True)      →   content: необов'язкове, <textarea>
created_at (auto_now_add)            →   немає в Meta.fields — у формі його немає
```

Цикл роботи: `NoteForm(data=...)` → `is_valid()` → `cleaned_data` (якщо все добре) або `errors` (якщо ні).

### Приклад 1.1. Які поля створила `ModelForm`

Порожня форма — `NoteForm()` — уже знає свої поля. Подивимось, що вона взяла з моделі.

In [ ]:
form = NoteForm()
for name, field in form.fields.items():
    print(f"{name:8} {type(field).__name__:10} required={field.required!s:5} "
          f"max_length={getattr(field, 'max_length', None)}  label={field.label!r}")

- `title` — `CharField` з `max_length=200` і `required=True`: у моделі немає `blank=True`;
- `content` — теж `CharField` форми (у моделі `TextField`), але `required=False`, бо в моделі `blank=True`;
- підписи — з `Meta.labels` у `forms.py`, а не з `verbose_name` моделі: явне налаштування форми важливіше.

### Приклад 1.2. Порожня й заповнена форма, HTML поля

Форма без даних — **unbound** («не прив'язана»): її показують на сторінці. Форма з `data=` — **bound**: її перевіряють. `form["title"]` — одне поле; надруковане, воно дає готовий HTML.

In [ ]:
empty = NoteForm()
filled = NoteForm(data={"title": "Здати звіт"})
print("прив'язана?", empty.is_bound, filled.is_bound)
print()
print(empty["title"])
print(empty["content"])

У HTML — усе з `forms.py` і моделі: `maxlength="200"` і `required` — з моделі, `class="form-control"`, `placeholder` і `autofocus` — з `widgets`. Браузер сам не пустить порожній заголовок, але на це покладатися не можна: запит можна надіслати й без браузера. Тому перевіряє **сервер** — `is_valid()`.

### Приклад 1.3. Правильні дані → `cleaned_data`

Дані з браузера — завжди **рядки**, часто з пробілами по краях. `is_valid()` запускає всі перевірки; лише після нього з'являється `cleaned_data` — очищені значення.

In [ ]:
form = NoteForm(data={"title": "  Здати звіт  ", "content": ""})
ok = form.is_valid()
data = form.cleaned_data

assert ok is True
assert data["title"] == "Здати звіт", "CharField прибирає пробіли на краях"
assert data["content"] == "", "content необов'язковий: blank=True у моделі"
print(ok, data)

`CharField` форми має `strip=True` за замовчуванням: пробіли на краях зникли, хоча ми їх надіслали.

### Приклад 1.4. Неправильні дані → `errors`

Порожній заголовок. `errors` — помилки **для кожного поля**; шаблон показує їх поруч із полем. Подивимось, що тоді в `cleaned_data`.

In [ ]:
form = NoteForm(data={"title": "", "content": "без заголовка"})
print("is_valid:", form.is_valid())
print("errors:", form.errors.get_json_data())
print("cleaned_data:", form.cleaned_data)

У `cleaned_data` лишилися **лише правильні** поля: `content` пройшов перевірку, `title` — ні. Тому `form.cleaned_data["title"]` у такій формі дав би `KeyError` — читай `cleaned_data` лише після `is_valid() == True`. Код помилки `required` зручний для тестів, текст — для людини (мова — з `LANGUAGE_CODE = "uk"`).

### Приклад 1.5. Задовгий заголовок: введене не губиться

201 символ — на один більше, ніж дозволяє модель. Перевіряє форма, ще до бази.

In [ ]:
bad = NoteForm(data={"title": "x" * 201, "content": "текст"})
bad.is_valid()
bad_fields = set(bad.errors)

assert bad_fields == {"title"}, "max_length=200 з моделі перевіряє форма"
for field, errors in bad.errors.items():
    print(field, "→", errors[0])
print("content у формі:", bad["content"].value())

Помилка лише в `title`. А `bad["content"].value()` — те, що користувач увів: сторінка з помилкою покаже форму **з його текстом**, і не доведеться набирати все заново.

**🔮 Прогноз:** Форма `NoteForm(data={'title': 'План'})` — без ключа `content` взагалі. Валідна?

<details>
<summary>Відповідь</summary>

Так. `content` у моделі `blank=True` → поле форми необов'язкове; `cleaned_data['content'] == ''`. А без `title` — помилка «Це поле обов'язкове.».

</details>

### Приклад 1.6. Перевіряємо прогноз

In [ ]:
plan = NoteForm(data={"title": "План"})
no_title = NoteForm(data={"content": "лише текст"})
print("План:", plan.is_valid(), plan.cleaned_data)
print("без title:", no_title.is_valid(), list(no_title.errors["title"]))

Ключа немає — для необов'язкового `CharField` це порожній рядок. Для обов'язкового — та сама помилка, що й для `""`.

### Приклад 1.7. `Meta.fields` — це захист

Зловмисник може надіслати будь-які поля, не лише ті, що є на сторінці: `id`, `created_at`… `ModelForm` бере з даних **лише** поля з `Meta.fields`. А `save(commit=False)` створює об'єкт моделі, але **не** записує його в базу — зручно, щоб щось дозаповнити перед збереженням (у рефакторингу 2 так додають власника).

In [ ]:
sneaky = NoteForm(data={"title": "Звичайна нотатка", "content": "", "id": "999", "created_at": "2000-01-01"})
print("is_valid:", sneaky.is_valid())
print("cleaned_data:", sneaky.cleaned_data)

draft = sneaky.save(commit=False)
print("об'єкт:", type(draft).__name__, "| pk:", draft.pk, "| created_at:", draft.created_at)
print("у базі є?", Note.objects.filter(title="Звичайна нотатка").exists())

`id` і `created_at` проігноровано мовчки: їх немає в `Meta.fields`. Після `commit=False` об'єкт є, але `pk` — `None`: у базу ще нічого не записано. Записав би лише `draft.save()`. Тому `fields` — завжди **явний** список: `fields = "__all__"` відкрив би для форми кожне нове поле моделі.

### Приклад 1.8. Власна перевірка одного поля: `clean_<поле>()`

Модель знає лише довжину. А правило «заголовок не з самих цифр» — наше. Таке правило пишуть методом `clean_title()`: Django викликає його **після** стандартних перевірок поля, у `self.cleaned_data["title"]` уже очищений рядок. Метод повертає значення (можна змінене) або кидає `forms.ValidationError`.

Тут ще й прибираємо подвійні пробіли всередині заголовка. Нова форма **успадковує** `NoteForm` — поля, віджети й підписи ті самі.

In [ ]:
class TidyNoteForm(NoteForm):
    def clean_title(self):
        title = " ".join(self.cleaned_data["title"].split())    # «Купити   хліб» → «Купити хліб»
        if title.isdigit():
            raise forms.ValidationError("Заголовок не може складатися лише з цифр.")
        return title

In [ ]:
for title in ["Купити    хліб", "2026", ""]:
    form = TidyNoteForm(data={"title": title})
    print(repr(title), "→", form.is_valid(), form.cleaned_data.get("title"), list(form.errors.get("title", [])))

- `"Купити    хліб"` пройшов і повернувся **зміненим** — у базу піде те, що повернув `clean_title()`;
- `"2026"` — наша помилка;
- `""` — стандартна помилка `required`: до `clean_title()` справа не дійшла, бо поле вже не пройшло перевірку.

### Приклад 1.9. Перевірка кількох полів разом: `clean()`

Правило «зміст не повторює заголовок» стосується **двох** полів — для нього метод `clean()`. Він працює після всіх `clean_<поле>()`; помилку прив'язуємо до конкретного поля через `add_error()`, щоб шаблон показав її біля `content`.

In [ ]:
class NoRepeatNoteForm(NoteForm):
    def clean(self):
        cleaned = super().clean()
        title, content = cleaned.get("title"), cleaned.get("content")
        if title and content and title.lower() == content.lower():
            self.add_error("content", "Зміст не має повторювати заголовок.")
        return cleaned

In [ ]:
same = NoRepeatNoteForm(data={"title": "Купити квитки", "content": "купити квитки"})
fine = NoRepeatNoteForm(data={"title": "Купити квитки", "content": "Київ — Львів"})
print("повтор:", same.is_valid(), same.errors.get_json_data())
print("різні:", fine.is_valid())

`cleaned.get(...)`, а не `cleaned[...]`: якщо `title` не пройшов власну перевірку, ключа в `cleaned_data` немає — `get` поверне `None`, і `clean()` не впаде.

### Приклад 1.10. Звичайна `Form` для пошуку (`GET`)

Не кожна форма зберігає модель. Пошук нічого не змінює — тому `method="get"`, а дані лежать в URL: `/notes/?q=молоко`. Для цього — `forms.Form` з полями, описаними вручну. `QueryDict` — саме такий об'єкт Django дає в `request.GET`.

In [ ]:
class NoteSearchForm(forms.Form):
    q = forms.CharField(required=False, max_length=50, label="Пошук")
    with_content = forms.BooleanField(required=False, label="Лише з текстом")


def search_notes(query_string):
    """Рядок запиту з URL → заголовки знайдених нотаток (або помилки форми)."""
    form = NoteSearchForm(QueryDict(query_string))
    if not form.is_valid():
        return form.errors.get_json_data()
    notes = Note.objects.filter(title__icontains=form.cleaned_data["q"])
    if form.cleaned_data["with_content"]:
        notes = notes.exclude(content="")
    return list(notes.values_list("title", flat=True))

In [ ]:
for query in ["q=молоко", "q=МОЛОКО", "q=", "q=Wi-Fi&with_content=on", "q=wi-fi", "q=" + "я" * 51]:
    print(f"{query[:20]:22} →", search_notes(query))

- `q=МОЛОКО` — **нічого**: SQLite порівнює без урахування регістру лише латиницю (`wi-fi` знайшов «Wi-Fi»), а для кирилиці `icontains` чутливий до регістру. PostgreSQL (урок 30) так не робить — це особливість SQLite, про яку варто пам'ятати в задачі 1;
- `q=` — порожній пошук: `icontains=""` збігається з усім, тож усі нотатки;
- `with_content=on` — так браузер надсилає позначений чекбокс; `BooleanField` перетворив `"on"` на `True`;
- задовгий запит — помилка форми замість запиту до бази. Ті самі `is_valid()` / `cleaned_data`, лише без моделі.

### Задача 1. Заголовок без дублікатів

Напиши `UniqueTitleNoteForm(NoteForm)` з методом `clean_title()`:

- якщо в базі вже є нотатка з таким самим заголовком **без урахування регістру** — `forms.ValidationError("Нотатка з такою назвою вже є.")`;
- порівнюй у Python (`.lower()`), а не `title__iexact`: SQLite ігнорує регістр лише для латиниці (приклад 1.10). Заголовки інших нотаток — `Note.objects.exclude(pk=...).values_list("title", flat=True)`;
- під час **редагування** нотатка не має конфліктувати сама з собою: виключи `self.instance.pk` (для нової форми це `None`, і `exclude` нічого не виключить);
- повертай заголовок.

Перевірка — у наступній клітинці.

In [ ]:
class UniqueTitleNoteForm(NoteForm):
    # BEGIN SOLUTION
    def clean_title(self):
        title = self.cleaned_data["title"]
        others = Note.objects.exclude(pk=self.instance.pk).values_list("title", flat=True)
        if any(other.lower() == title.lower() for other in others):
            raise forms.ValidationError("Нотатка з такою назвою вже є.")
        return title
    # END SOLUTION

In [ ]:
notes = seed()
twin = UniqueTitleNoteForm(data={"title": "купити МОЛОКО"})
assert twin.is_valid() is False, "«купити МОЛОКО» вже є як «Купити молоко» — форма має не пройти"
assert twin.errors["title"] == ["Нотатка з такою назвою вже є."]
assert UniqueTitleNoteForm(data={"title": "Купити хліб"}).is_valid(), "нова назва — без помилок"
milk = notes["Купити молоко"]
same = UniqueTitleNoteForm(data={"title": "Купити молоко", "content": "3 л"}, instance=milk)
assert same.is_valid(), "редагування не конфліктує саме з собою: виключи self.instance.pk"
print("✅ Задача 1 розв'язана")

---
## 2. Що view передає в шаблон

```
GET /notes/ → urls.py → note_list(request) → Note.objects.all() → render(шаблон, {"notes": ...}) → HTML
```

Тестовий клієнт повертає не лише HTML, а й **контекст шаблону** — словник, який view передав у `render()`, і список шаблонів, з яких зібрано сторінку.

### Приклад 2.1. Контекст сторінки списку

In [ ]:
notes = seed()
response = client.get("/notes/")
titles = [note.title for note in response.context["notes"]]

assert titles == ["Пароль від Wi-Fi", "Вивчити Django Forms", "Купити молоко"], "Meta.ordering = ['-created_at']: нові — першими"
assert response.templates[0].name == "hello_app/note_list.html"
assert "3 нотаток" in response.content.decode(), "{{ notes|length }} у шаблоні"
print(response.status_code, titles)

`response.context["notes"]` — той самий `QuerySet`, що view поклав у словник. А в HTML шаблон надрукував `{{ notes|length }}` → «3 нотаток».

### Приклад 2.2. З яких шаблонів зібрано сторінку

In [ ]:
names = [template.name for template in response.templates]
print("перші два:", names[:2])
print("усього шаблонів:", len(names), "| за пакетами:", Counter(name.split("/")[0] for name in names[2:]))

Першим — `note_list.html`, другим — `base.html`: сторінка починається з `{% extends 'hello_app/base.html' %}` і заповнює лише свої блоки. Змінити navbar — змінити один `base.html` для всіх сторінок.

Решта — не наші шаблони: панель Django Debug Toolbar (`DEBUG = True`, а тестовий клієнт приходить з `127.0.0.1`, що є в `INTERNAL_IPS`, — панель вбудовується в кожну сторінку) і дрібні шаблони віджетів форм Django (`django/forms/...`), з яких складаються поля панелі. На сервері в продакшні `DEBUG = False` — і панелі немає.

### Приклад 2.3. `{% extends %}` і `{% block %}` у мініатюрі

Щоб побачити механізм без Bootstrap, зберемо крихітний «рушій шаблонів» з двома шаблонами в пам'яті (`locmem`-завантажувач). `{{ block.super }}` — вміст того самого блоку з батьківського шаблону.

In [ ]:
MINI_TEMPLATES = {
    "base.html": (
        "<title>{% block title %}Нотатки{% endblock %}</title>\n"
        "<nav>Bootstrap Notes</nav>\n"
        "<main>{% block content %}{% endblock %}</main>"
    ),
    "list.html": (
        "{% extends 'base.html' %}"
        "{% block title %}Список — {{ block.super }}{% endblock %}"
        "{% block content %}{% for note in notes %}<p>{{ note.title }}</p>{% empty %}<p>Нотаток ще немає</p>{% endfor %}{% endblock %}"
    ),
}
mini_engine = Engine(loaders=[("django.template.loaders.locmem.Loader", MINI_TEMPLATES)])

In [ ]:
page = mini_engine.get_template("list.html")
print(page.render(Context({"notes": Note.objects.all()[:2]})))
print("---")
print(page.render(Context({"notes": []})))

`list.html` не має ні `<title>`, ні `<nav>` — вони прийшли з `base.html`. Заголовок вкладки — «Список — Нотатки»: свій текст + `block.super`. `{% empty %}` спрацьовує, коли список порожній.

### Приклад 2.4. Шаблон екранує HTML

Користувач може ввести в заголовок що завгодно — навіть `<script>`. Створимо таку нотатку **через форму** і подивимось на сторінку списку.

In [ ]:
evil = NoteForm(data={"title": "<script>alert('hi')</script>", "content": ""})
assert evil.is_valid(), "для форми це просто текст до 200 символів"
evil.save()
html = client.get("/notes/").content.decode()
print("сирий <script> у сторінці?", "<script>alert" in html)
print("екранований:", "&lt;script&gt;alert(&#x27;hi&#x27;)&lt;/script&gt;" in html)

Форма пропустила рядок — це просто текст. Але `{{ note.title }}` у шаблоні **екранує** `<`, `>` і лапки: браузер покаже текст, а не виконає скрипт. Це захист від XSS «з коробки»; вимикає його лише фільтр `|safe` — для даних користувача його не ставлять.

### Приклад 2.5. Адреси за іменем: `reverse` і `resolve`

У шаблонах посилання пишуть як `{% url 'hello_app:note_detail' pk=note.pk %}`, а в Python — `reverse(...)`. Обернена дія — `resolve`: яка функція обробить адресу і з якими аргументами.

In [ ]:
milk = notes["Купити молоко"]
print(reverse("hello_app:note_list"))
print(reverse("hello_app:note_detail", kwargs={"pk": milk.pk}))
print(reverse("hello_app:note_edit", args=[milk.pk]))

match = resolve(f"/notes/{milk.pk}/delete/")
print(match.view_name, match.func.__name__, match.kwargs)

Адреса будується з `urls.py`: змінимо там `notes/` на `my-notes/` — і `reverse`, і `{% url %}` дадуть нову адресу, а шаблони не доведеться правити. `<int:pk>` у маршруті перетворив `"…"` з адреси на **число** в `kwargs`.

### Приклад 2.6. Сторінка нотатки і `404`

`note_detail` шукає нотатку через `get_object_or_404`: немає — `404 Not Found`, а не `500` від винятку `DoesNotExist`.

In [ ]:
detail = client.get(f"/notes/{milk.pk}/")
print("є:", detail.status_code, detail.context["note"].title)
print("немає:", client.get("/notes/999999/").status_code)
print("не число:", client.get("/notes/abc/").status_code)

Обидва `404` різні за природою: `/notes/999999/` маршрут знайшов, а нотатки немає (`get_object_or_404`); `/notes/abc/` не підійшов до жодного маршруту — `<int:pk>` приймає лише цифри. Журнал Django записав обидва як `Not Found`.

---
## 3. Створення: помилки і PRG

`note_create` має три гілки:

```
GET  /notes/new/                  → 200, порожня форма
POST /notes/new/ з помилками      → 200, та сама форма: введене + помилки
POST /notes/new/ без помилок      → form.save() (INSERT) → 302 Location: /notes/
                                     браузер сам робить GET /notes/ — F5 повторить лише цей GET
```

Третя гілка — шаблон **PRG** (Post / Redirect / Get).

### Приклад 3.1. `GET`: порожня форма

In [ ]:
notes = seed()
page = client.get("/notes/new/")
html = page.content.decode()
print(page.status_code, "| прив'язана форма?", page.context["form"].is_bound)
print("csrf-токен у формі:", 'name="csrfmiddlewaretoken"' in html)
print("кнопка:", page.context["action"], "| заголовок:", page.context["title"])

`GET` нічого не перевіряє: форма не прив'язана, помилок немає. У HTML уже є прихований `csrfmiddlewaretoken` — його вставив `{% csrf_token %}` (розділ 5). `action` і `title` — з контексту view: один шаблон `note_form.html` служить і створенню, і редагуванню.

### Приклад 3.2. `POST` з помилкою — та сама сторінка

In [ ]:
before = Note.objects.count()
bad_response = client.post("/notes/new/", {"title": "", "content": "текст"})

assert bad_response.status_code == 200, "неправильні дані — та сама сторінка з формою"
assert "title" in bad_response.context["form"].errors
assert bad_response.context["form"]["content"].value() == "текст", "введене не губиться"
assert Note.objects.count() == before, "нічого не збережено"
print(bad_response.status_code, dict(bad_response.context["form"].errors))

Статус `200`, а не `400`: для браузера це звичайна сторінка, просто з помилками біля полів. У базі — нічого нового.

### Приклад 3.3. `POST` без помилок — redirect і повідомлення

In [ ]:
created = client.post("/notes/new/", {"title": "Купити квитки", "content": "Київ — Львів"})
page = client.get(created["Location"])
shown = [str(message) for message in page.context["messages"]]

assert created.status_code == 302 and created["Location"] == "/notes/", "PRG: redirect на список"
assert shown == ['Нотатку "Купити квитки" успішно створено!']
assert Note.objects.filter(title="Купити квитки").count() == 1
print(created.status_code, created["Location"], shown)

Відповідь на `POST` — лише `302` з заголовком `Location`; HTML у ній немає. Повідомлення view поклав у сесію (`messages.success`), а показала його **наступна** сторінка — та, на яку перенаправили.

**🔮 Прогноз:** Що покаже `client.get(created['Location'])` вдруге — те саме повідомлення?

<details>
<summary>Відповідь</summary>

Ні: повідомлення «на одну сторінку». Після першого показу воно зникає, а повторний `GET` нічого не створює й нічого не повідомляє.

</details>

### Приклад 3.4. Перевіряємо прогноз: повідомлення одноразове

In [ ]:
again = client.get(created["Location"])
print("повідомлення вдруге:", [str(message) for message in again.context["messages"]])
print("нотаток «Купити квитки»:", Note.objects.filter(title="Купити квитки").count())

Порожньо, і дубля немає. Саме це й дає PRG: F5 на сторінці списку повторює безпечний `GET`.

### Приклад 3.5. `follow=True`: пройти redirect, як браузер

Тестовий клієнт може сам іти за перенаправленнями. `redirect_chain` — шлях, яким він пройшов.

In [ ]:
final = client.post("/notes/new/", {"title": "Подзвонити мамі"}, follow=True)
print("ланцюжок:", final.redirect_chain)
print("фінальна сторінка:", final.status_code, final.templates[0].name)
print("повідомлення:", [str(message) for message in final.context["messages"]])

Один крок `302 → /notes/`, а фінальна відповідь — уже сторінка списку з повідомленням. Так зручно перевіряти «що побачить користувач після кнопки».

### Приклад 3.6. Рівні повідомлень → класи Bootstrap

`messages.success`, `messages.warning`… — це **рівні**. `MESSAGE_TAGS` у `settings.py` перекладає рівень у клас Bootstrap: шаблон пише `alert-{{ message.tags }}`. Подивимось на переклад для кожного рівня.

In [ ]:
for level_name in ["DEBUG", "INFO", "SUCCESS", "WARNING", "ERROR"]:
    level = getattr(message_levels, level_name)
    print(f"{level_name:8} рівень {level:2} → alert-{Message(level, 'текст').tags}")

`ERROR` → `danger`: у Bootstrap червоний alert називається саме так, тому без `MESSAGE_TAGS` помилка мала б клас `alert-error`, якого Bootstrap не знає. А `messages.info(...)` дасть блакитний `alert-info` — відповідь на «Зміни приклад» зі сторінки уроку.

---
## 4. Редагування і видалення

Редагування — той самий PRG, але форма отримує `instance=note`: показує поточні значення, а `save()` робить `UPDATE`, а не `INSERT`. Видалення: `GET` — сторінка «Точно видалити?», `POST` — видалення.

### Приклад 4.1. Форма редагування вже заповнена

In [ ]:
notes = seed()
milk = notes["Купити молоко"]
edit_page = client.get(f"/notes/{milk.pk}/edit/")
form = edit_page.context["form"]
print(edit_page.status_code, "| title:", form["title"].value(), "| content:", form["content"].value())
print("instance:", form.instance.pk == milk.pk, "| кнопка:", edit_page.context["action"])

`NoteForm(instance=note)` заповнила поля з об'єкта: користувач бачить, що змінює. `form.instance` — та сама нотатка з бази.

### Приклад 4.2. `POST` редагування — `UPDATE`

In [ ]:
count_before = Note.objects.count()
edited = client.post(f"/notes/{milk.pk}/edit/", {"title": "Купити молоко і хліб", "content": milk.content})
milk.refresh_from_db()

assert edited.status_code == 302 and edited["Location"] == f"/notes/{milk.pk}/", "після редагування — на деталі"
assert milk.title == "Купити молоко і хліб"
assert Note.objects.count() == count_before, "редагування не створює нову нотатку"
print(edited.status_code, edited["Location"], "|", milk.title)

`pk` той самий, нотаток стільки ж — `save()` оновив рядок. `refresh_from_db()` потрібен, бо змінив базу інший об'єкт (у view), а наша змінна `milk` ще тримала старі значення.

### Приклад 4.3. Що саме змінилося: `changed_data`

Форма з `instance=` порівнює надіслане з початковим. Це знадобиться, наприклад, щоб не зберігати нічого, коли користувач нічого не змінив, або записати в журнал, які поля правили.

In [ ]:
wifi = notes["Пароль від Wi-Fi"]
nothing = NoteForm(data={"title": wifi.title, "content": wifi.content}, instance=wifi)
new_password = NoteForm(data={"title": wifi.title, "content": "tea2027"}, instance=wifi)
print("нічого:", nothing.has_changed(), nothing.changed_data)
print("пароль:", new_password.has_changed(), new_password.changed_data)

`has_changed()` — чи змінилося хоч щось; `changed_data` — список змінених полів. Перевірки не потрібно: порівнюються значення.

### Приклад 4.4. Видалення: `GET` питає, `POST` видаляє

In [ ]:
url = f"/notes/{wifi.pk}/delete/"
confirm_code = client.get(url).status_code
still_there = Note.objects.filter(pk=wifi.pk).exists()
deleted = client.post(url, follow=True)

assert confirm_code == 200 and still_there, "GET лише питає"
assert deleted.redirect_chain == [("/notes/", 302)]
assert not Note.objects.filter(pk=wifi.pk).exists()
message = list(deleted.context["messages"])[0]
print("GET:", confirm_code, still_there, "| POST →", deleted.redirect_chain, "|", message, "→ alert-" + message.tags)
print("ще раз:", client.post(url).status_code)

`GET` на адресу видалення нічого не змінює: посилання відкривають пошукові роботи й попереднє завантаження браузера. Повідомлення — рівня `warning` (жовте). Повторний `POST` — `404`: нотатки вже немає, `get_object_or_404` спрацював.

### Приклад 4.5. `@require_POST`: лише `POST`, інакше `405`

`note_delete` перевіряє метод сам (`if request.method == 'POST'`). Для дій **без** сторінки підтвердження Django має декоратор: `@require_POST` відповідає `405 Method Not Allowed` на будь-що, крім `POST`. Перевіримо на маленькому view, викликаючи його напряму з `RequestFactory`.

In [ ]:
@require_POST
def ping(request):
    return HttpResponse("pong")

In [ ]:
for method in ["get", "post", "put"]:
    response = ping(getattr(factory, method)("/ping/"))
    print(method.upper(), "→", response.status_code, response.get("Allow", ""))

`GET` і `PUT` — `405` з заголовком `Allow: POST`: сервер підказує, який метод дозволено. До тіла функції справа не дійшла. `RequestFactory` створює лише об'єкт запиту — без маршрутів і middleware, — тож view викликаємо як звичайну функцію.

### Задача 2. Кнопка «Дублювати»

Напиши view `note_duplicate(request, pk)` (декоратор `@require_POST` уже стоїть):

- знайди нотатку через `get_object_or_404(Note, pk=pk)`;
- створи **нову** нотатку з заголовком `"<старий заголовок> (копія)"` і тим самим змістом;
- поверни redirect на сторінку **нової** нотатки: `redirect("hello_app:note_detail", pk=...)` — це PRG.

Перевірка — у наступній клітинці: вона викликає view напряму, як у прикладі 4.5.

In [ ]:
@require_POST
def note_duplicate(request, pk):
    # BEGIN SOLUTION
    note = get_object_or_404(Note, pk=pk)
    copy = Note.objects.create(title=f"{note.title} (копія)", content=note.content)
    return redirect("hello_app:note_detail", pk=copy.pk)
    # END SOLUTION

In [ ]:
source = Note.objects.create(title="Здати звіт", content="до п'ятниці")
response = note_duplicate(factory.post(f"/notes/{source.pk}/duplicate/"), pk=source.pk)
assert response is not None, "view має повернути відповідь — redirect на нову нотатку"
assert response.status_code == 302, "PRG: після створення — redirect (302)"
copy = Note.objects.get(title="Здати звіт (копія)")
assert response["Location"] == f"/notes/{copy.pk}/", "redirect саме на сторінку копії"
assert copy.pk != source.pk and copy.content == "до п'ятниці"
assert note_duplicate(factory.get(f"/notes/{source.pk}/duplicate/"), pk=source.pk).status_code == 405
try:
    note_duplicate(factory.post("/notes/999999/duplicate/"), pk=999999)
    raise AssertionError("для неіснуючої нотатки — Http404 через get_object_or_404")
except Http404:
    pass
print("✅ Задача 2 розв'язана")

---
## 5. CSRF

**CSRF** (Cross-Site Request Forgery): чужий сайт змушує твій браузер надіслати `POST` на наш сайт — з твоїми cookie, від твого імені. Захист: `{% csrf_token %}` кладе у форму секретний токен, а `CsrfViewMiddleware` відхиляє `POST` без правильного токена. Чужий сайт токена не знає.

Звичайний тестовий клієнт CSRF не перевіряє (зручно для тестів). `Client(enforce_csrf_checks=True)` поводиться як справжній сервер.

### Приклад 5.1. Без токена — `403`

In [ ]:
strict = Client(enforce_csrf_checks=True)
code = strict.post("/notes/new/", {"title": "Без токена"}).status_code

assert code == 403
assert not Note.objects.filter(title="Без токена").exists()
print(code)

`403 Forbidden`, а журнал пояснив причину — `CSRF cookie not set`: запит прийшов не зі сторінки нашої форми.

### Приклад 5.2. З токеном — як справжній браузер

Браузер спершу відкриває сторінку форми (`GET`): сервер ставить cookie `csrftoken` і вставляє токен у форму. Потім браузер надсилає форму разом із токеном. Повторимо це тим самим «суворим» клієнтом.

In [ ]:
strict.get("/notes/new/")                                   # як браузер: спершу сторінка форми
token = strict.cookies["csrftoken"].value
with_token = strict.post("/notes/new/", {"title": "З токеном", "csrfmiddlewaretoken": token})
print("cookie csrftoken:", len(token), "символів")
print("POST з токеном:", with_token.status_code, with_token["Location"])
print("збережено:", Note.objects.filter(title="З токеном").exists())

Той самий `POST`, але з токеном — `302` і нова нотатка. Чужий сайт може змусити браузер надіслати cookie, але **прочитати** токен зі сторінки нашого сайту не може — тому підробити такий запит не вийде.

---
## 6. Знайди помилку

```python
def note_create_buggy(request):
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
    return render(request, "hello_app/note_form.html", {"form": form, "title": "Нова нотатка", "action": "Створити"})
```

Після успішного `POST` — сторінка `200`: F5 надішле форму ще раз і створить дубль.

### Приклад 6.1. Відтворюємо помилку: «F5» після `POST`

Визначимо view з помилкою й надішлемо **той самий** `POST` двічі — так робить браузер, коли після `POST` натиснути F5 і погодитися «надіслати повторно».

In [ ]:
def note_create_buggy(request):
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
    return render(request, "hello_app/note_form.html", {"form": form, "title": "Нова нотатка", "action": "Створити"})

In [ ]:
before = Note.objects.filter(title="Оплатити світло").count()
for attempt in (1, 2):                                       # другий раз — це F5
    response = note_create_buggy(factory.post("/notes/new/", {"title": "Оплатити світло"}))
    print("спроба", attempt, "→", response.status_code)
print("дублів:", Note.objects.filter(title="Оплатити світло").count() - before)

Обидві відповіді — `200` зі сторінкою форми, і в базі **дві** однакові нотатки. Браузер не знає, що форма вже збережена: остання дія на сторінці — `POST`, і F5 повторює саме його.

### Задача 3. Виправ: PRG

Напиши `note_create_fixed(request)` за шаблоном PRG: успіх → `redirect("hello_app:note_list")`, інакше — форма з помилками (той самий `render`, що в `note_create_buggy`).

Перевірка — у наступній клітинці.

In [ ]:
def note_create_fixed(request):
    # BEGIN SOLUTION
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
        return redirect("hello_app:note_list")
    return render(request, "hello_app/note_form.html", {"form": form, "title": "Нова нотатка", "action": "Створити"})
    # END SOLUTION

In [ ]:
response = note_create_fixed(factory.post("/notes/new/", {"title": "Оплатити інтернет"}))
assert response is not None and response.status_code == 302, "після успішного POST — redirect (302), а не сторінка 200"
assert response["Location"] == "/notes/"
assert Note.objects.filter(title="Оплатити інтернет").exists()
assert note_create_fixed(factory.post("/notes/new/", {"title": ""})).status_code == 200, "помилки — та сама форма"
assert note_create_fixed(factory.get("/notes/new/")).status_code == 200, "GET — порожня форма"
print("✅ Задача 3 розв'язана")

---
## 7. Рефакторинг 2: `crispy_notes_project`

Що змінилося в рефакторингу 2 (докладно — на [сторінці уроку](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/#refactor-2)):

```
рефакторинг 1                                  рефакторинг 2
NoteForm: widgets з form-control              NoteForm: FormHelper + Layout, {% crispy form %} у шаблоні
base.html → сторінка                           base.html → layouts/dashboard.html → сторінка
form.save() у view                             services.create_note(...) у view
кожен може все                                 @login_required; NoteForm(user=...) показує лише свої записники
sidebar — немає                                context processor sidebar_context для кожного render()
```

Django налаштовується **один раз на процес**: у цьому ядрі вже працює `django_bootstrap_project`, і другий проєкт з тими самими іменами модулів (`hello_project`, `hello_app`) сюди не завантажиш. Тому код рефакторингу 2 запускаємо в **окремому процесі** — `python manage.py shell -c "<код>"` у папці `crispy_notes_project`, як ти робив би в терміналі. Кожен скрипт — константа-рядок (визначення), запуск — у наступній клітинці.

In [ ]:
CRISPY_DIR = os.path.join(LESSON_DIR, "crispy_notes_project")


def run_in_crispy(script):
    """Виконати код у Django-shell проєкту crispy_notes_project (окремий процес) і показати вивід."""
    result = subprocess.run([sys.executable, "manage.py", "shell", "-v", "0", "-c", script],
                            cwd=CRISPY_DIR, capture_output=True, text=True)
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr[-3000:])
    return result

### Приклад 7.1. База другого проєкту: користувачі й записники

У рефакторингу 2 кожна нотатка, записник і тег мають власника (`user`). Створимо Олену з двома записниками й тегом і Боба з одним записником.

In [ ]:
migrated = subprocess.run([sys.executable, "manage.py", "migrate", "-v", "0"], cwd=CRISPY_DIR, capture_output=True, text=True)
print("migrate:", "OK" if migrated.returncode == 0 else migrated.stderr[-1000:])

CRISPY_SEED = r"""
from django.contrib.auth.models import User
from hello_app.models import Notebook, Tag

User.objects.filter(username__in=["olena", "bob"]).delete()   # разом з їхніми нотатками й записниками
olena = User.objects.create_user("olena", password="pass-12345")
bob = User.objects.create_user("bob", password="pass-12345")
Notebook.objects.create(user=olena, title="Навчання", color="#2e7d32")
Notebook.objects.create(user=olena, title="Подорожі", color="#e65100")
Notebook.objects.create(user=bob, title="Записник Боба")
Tag.objects.create(user=olena, name="django")
print("записників:", Notebook.objects.count(), "| Олени:", olena.notebooks.count(), "| тегів Олени:", olena.tags.count())
"""

In [ ]:
run_in_crispy(CRISPY_SEED);

`User.objects.filter(...).delete()` на початку — щоб скрипт можна було запускати знову: користувачі зникають разом з усім, що їм належить (`on_delete=CASCADE`).

### Приклад 7.2. `{% crispy form %}`: розмітка з `Layout`

Шаблон форми тепер — один тег. Відрендеримо його самі й подивимось, що crispy згенерував з `FormHelper` і `Layout` у `forms.py`: тег `<form>`, CSRF-токен, `fieldset`-и, сітку колонок і класи Bootstrap.

In [ ]:
CRISPY_RENDER = r"""
import re
from django.contrib.auth.models import User
from django.template import engines
from django.test import RequestFactory
from hello_app.forms import NoteForm, NotebookForm

olena = User.objects.get(username="olena")
request = RequestFactory().get("/notes/new/")
request.user = olena
page = engines["django"].from_string("{% load crispy_forms_tags %}{% crispy form %}")

html = page.render({"form": NoteForm(user=olena)}, request)
print("<form>:", re.findall(r"<form[^>]*>", html))
print("csrf-токен:", 'name="csrfmiddlewaretoken"' in html)
print("fieldset:", re.findall(r"<legend[^>]*>\s*(.*?)\s*</legend>", html))
print("колонки:", re.findall(r'class="(col-md-\d+)', html))
print("form-control:", html.count("form-control"), "| form-select:", html.count("form-select"))

notebook_html = page.render({"form": NotebookForm()}, request)
print("NotebookForm колір:", re.findall(r'<input type="color"[^>]*name="(\w+)"', notebook_html))
"""

In [ ]:
run_in_crispy(CRISPY_RENDER);

- `<form id="note-form" method="post">` — з `helper.form_id` і `helper.form_method`; токен CSRF crispy вставив сам;
- три `fieldset` з `Layout` — «Основна інформація», «Зміст нотатки», «Теги та параметри»;
- `col-md-4` + `col-md-8` — `Row(Column(...), Column(...))`: пріоритет і записник в одному рядку;
- класи `form-control` / `form-select` у `forms.py` ніде не написані — їх додав шаблонний пакет `bootstrap5`;
- у `NotebookForm` лишився єдиний `widgets` — щоб поле `color` стало `<input type="color">`.

Перевірка даних не змінилася: `FormHelper` впливає лише на **вигляд**.

### Приклад 7.3. Вхід, фільтр записників, context processor

Перевіримо поведінку сторінок тестовим клієнтом у другому проєкті — ті самі запити, що на сторінці уроку.

In [ ]:
CRISPY_CLIENT = r"""
from django.test import Client
from django.test.utils import setup_test_environment
from hello_app.models import Note

setup_test_environment()
client = Client()

r = client.get("/notes/")
print("анонім        →", r.status_code, r["Location"])

client.login(username="olena", password="pass-12345")
r = client.get("/notes/new/")
print("форма         →", r.status_code, 'id="note-form"' in r.content.decode())
print("записники     →", [nb.title for nb in r.context["form"].fields["notebook"].queryset])
print("sidebar       →", [(nb.title, nb.note_count) for nb in r.context["sidebar_notebooks"]])

r = client.post("/notes/new/", {"title": "", "priority": "2"})
print("помилка       →", r.status_code, dict(r.context["form"].errors))

r = client.post("/notes/new/", {"title": "Здати проєкт", "priority": "3", "is_pinned": "on"})
note = Note.objects.get(title="Здати проєкт")
print("створено      →", r.status_code, r["Location"] == f"/notes/{note.pk}/", note.user, note.is_pinned)
"""

In [ ]:
run_in_crispy(CRISPY_CLIENT);

- анонім — `302` на сторінку входу з `?next=/notes/`: це `@login_required` (вхід і реєстрація — урок 41);
- у списку записників форми — **лише** записники Олени: `NoteForm(user=...)` фільтрує `queryset`;
- `sidebar_notebooks` є в контексті, хоча view його не передавав, — його додав context processor; `note_count` — кількість нотаток, яку порахував selector;
- помилки валідації ті самі, що в рефакторингу 1; успіх — redirect на сторінку нової нотатки, власник — Олена, нотатка закріплена.

### Приклад 7.4. Чужий записник не підставиш

Список записників у формі — не лише вигляд. Спробуємо надіслати `POST` із записником **Боба**, ніби підробили HTML-форму.

In [ ]:
CRISPY_FOREIGN = r"""
from django.contrib.auth.models import User
from hello_app.forms import NoteForm
from hello_app.models import Notebook

olena = User.objects.get(username="olena")
bob_notebook = Notebook.objects.get(title="Записник Боба")
own_notebook = Notebook.objects.get(title="Навчання")
for notebook in (bob_notebook, own_notebook):
    form = NoteForm(data={"title": "Нотатка в записник", "priority": "1", "notebook": notebook.pk}, user=olena)
    print(notebook.title, "→", form.is_valid(), list(form.errors.get("notebook", [])))
"""

In [ ]:
run_in_crispy(CRISPY_FOREIGN);

Записник Боба — помилка «Зробить коректний вибір. Такого варіанту нема серед доступних.» (так звучить переклад Django): `ModelChoiceField` перевіряє, що значення є в **його** `queryset`, а там лише записники Олени. Захист працює на сервері, хоч би що прийшло з браузера.

### Приклад 7.5. Context processor — звичайна функція

`sidebar_context(request)` — функція «запит → словник». Django викликає її для кожного `render()`, але її можна викликати й напряму.

In [ ]:
CRISPY_SIDEBAR = r"""
from django.contrib.auth.models import AnonymousUser, User
from django.test import RequestFactory
from hello_app.context_processors import sidebar_context

request = RequestFactory().get("/")
for user in (AnonymousUser(), User.objects.get(username="olena")):
    request.user = user
    context = sidebar_context(request)
    print(user, "→", sorted(context), [str(nb) for nb in context["sidebar_notebooks"]])
"""

In [ ]:
run_in_crispy(CRISPY_SIDEBAR);

Для аноніма — порожні списки (ані запиту до бази), для Олени — її записники. Ключі ті самі: шаблонам `layouts/dashboard.html` не треба перевіряти, чи вони є. Підключено функцію одним рядком у `TEMPLATES["OPTIONS"]["context_processors"]`.

### Приклад 7.6. Тихий баг: прапорець, що не дійшов до сервісу

Зі сторінки уроку («Знайди помилку»): view перекладає `cleaned_data` в `services.create_note(...)` **вручну**, поле за полем. Пропусти одне поле — і воно мовчки загубиться. Відтворимо обидва варіанти виклику на тій самій формі.

In [ ]:
CRISPY_PINNED = r"""
from django.contrib.auth.models import User
from hello_app import services
from hello_app.forms import NoteForm

olena = User.objects.get(username="olena")
form = NoteForm(data={"title": "Закріпи мене", "priority": "2", "is_pinned": "on"}, user=olena)
print("форма валідна:", form.is_valid(), "| cleaned_data['is_pinned'] =", form.cleaned_data["is_pinned"])
data = form.cleaned_data

forgot = services.create_note(user=olena, title=data["title"], content=data["content"],
                              priority=data["priority"], notebook=data["notebook"])
passed = services.create_note(user=olena, title=data["title"], content=data["content"],
                              priority=data["priority"], notebook=data["notebook"], is_pinned=data["is_pinned"])
print("без is_pinned  →", forgot.is_pinned)
print("з is_pinned    →", passed.is_pinned)
"""

In [ ]:
run_in_crispy(CRISPY_PINNED);

Форма прийняла прапорець (`True`), помилки немає — але без `is_pinned=` у виклику сервісу нотатка не закріплена. Кожне нове поле форми треба додати в **трьох** місцях: `Meta.fields`, виклик сервісу, сигнатура сервісу. Тест `test_is_pinned_is_saved_on_create` у `hello_app/tests.py` проєкту ловить саме це.

---
## Самоперевірка

1. Коли форма `method="get"`, а коли `method="post"`?
2. Звідки `NoteForm` знає, що `title` не довший за 200 символів, а `content` необов'язковий?
3. Що буде в `cleaned_data`, якщо одне з двох полів не пройшло перевірку?
4. Чим `clean_title()` відрізняється від `clean()`?
5. Навіщо `save(commit=False)`?
6. Опиши PRG. Яку проблему він розв'язує?
7. Чому видалення — лише `POST`?
8. Від чого захищає `{% csrf_token %}`?
9. Що зникає з `forms.py` і шаблону при переході на crispy, а що лишається незмінним?

<details>
<summary>Відповіді</summary>

1. `GET` — пошук і фільтри (нічого не змінює, дані в URL); `POST` — створення, зміна, видалення.
2. З моделі: `ModelForm` будує поля з `CharField(max_length=200)` і `TextField(blank=True)`.
3. Лише правильні поля; неправильного ключа немає — тому `cleaned_data` читають після `is_valid() == True`, а в `clean()` — через `get`.
4. `clean_title()` перевіряє одне поле й повертає його значення; `clean()` — кілька полів разом, помилку прив'язують до поля через `add_error()`.
5. Отримати об'єкт моделі без запису в базу — щоб дозаповнити поля (наприклад, власника) і лише потім `save()`.
6. Post / Redirect / Get: після успішного `POST` — `302`, браузер робить `GET`. F5 повторить лише `GET`, дублів немає.
7. `GET` не має змінювати дані: посилання відкривають роботи, попереднє завантаження, розширення. `POST` захищений CSRF-токеном.
8. Від CSRF: чужий сайт не знає секретного токена сторінки, тож його підроблений `POST` отримає `403`.
9. Зникають `widgets` з класами Bootstrap і ручний HTML форми; лишаються поля, перевірки (`is_valid()`, `cleaned_data`) і views.

</details>

## Далі

- Книга курсу: [Урок 35](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/) — рефакторинг 1 (Bootstrap CRUD) і рефакторинг 2 (crispy dashboard), діфи й скріншоти.
- Запусти проєкт: `python manage.py runserver` у папці `django_bootstrap_project` → `http://127.0.0.1:8000/notes/`; тести — `python manage.py test`.
- Спробуй рефакторинг 2 сам: переведи `note_form.html` цього проєкту на `{% crispy form %}` — завдання «Спробуй самостійно» на сторінці уроку.
- **Урок 36** — наступний рефакторинг: REST API до нотаток (DRF).